# SciML Sensitivity Optimizers + Symbolic recovery in Julia

## Introduction

In this tutorial, we will look at:

(a) How optimizers can be chained to solve an inverse problem efficiently with Optimization.jl and SciMLSensitivity.jl.

(b) Symbolic recovery in Julia, with DataDrivenDiffEq.jl.

## Libraries

In [ ]:
Pollis.packages("Lux", "OrdinaryDiffEq", "SciMLSensitivity", "Optimization", "OptimizationOptimisers", "OptimizationOptimJL", "ComponentArrays", "Mooncake", "DataDrivenDiffEq", "DataDrivenSparse", "Plots")

In [ ]:
using Lux, OrdinaryDiffEq, SciMLSensitivity, Optimization, OptimizationOptimisers, OptimizationOptimJL, ComponentArrays, Mooncake
using DataDrivenDiffEq, DataDrivenSparse, LinearAlgebra, Statistics, Random, Plots

rng = Xoshiro(1234)

## Data Generation
Generate data for the predator-prey (Lotka-Volterra) model.

In [ ]:
function lotka!(du, u, p, t)
    alpha, beta, gamma, delta = p
    du[1] = alpha * u[1] - beta * u[2] * u[1]
    du[2] = gamma * u[1] * u[2] - delta * u[2]
end

# Define the experimental parameters
tspan = (0.0, 3.0)
u0 = [0.44249296, 4.6280594]
p_ = [1.3, 0.9, 0.8, 1.8]
prob = ODEProblem(lotka!, u0, tspan, p_)
solution = solve(prob, Vern7(); abstol = 1e-12, reltol = 1e-12, saveat = 0.1)

# Ideal data
X = Array(solution)
t = solution.t

# Add noise in terms of the mean
x_bar = mean(X; dims = 2)
noise_magnitude = 5e-3
X_n = X .+ (noise_magnitude * x_bar) .* randn(rng, size(X))

plot(solution; alpha = 0.75, color = :black, label = ["True Data" nothing])
scatter!(t, transpose(X_n); color = :red, label = ["Noisy Data" nothing])

## Define a simple Universal Differential Equation (UDE)
The linear terms, alpha u1 and -delta u2, are known; a network learns the interactions.

In [ ]:
# Gaussian RBF as activation
rbf(x) = exp.(-(x .^ 2))

# Multilayer FeedForward
U = Chain(Dense(2 => 5, rbf), Dense(5 => 5, rbf), Dense(5 => 5, rbf), Dense(5 => 2))
# Get the initial parameters and state variables of the model
p, st = f64(Lux.setup(rng, U))

# Define the hybrid model
function ude_dynamics(u, p, t, p_true)
    u_hat = first(U(u, p, st))  # Network prediction
    return [p_true[1] * u[1] + u_hat[1], -p_true[4] * u[2] + u_hat[2]]
end

# Closure with the known parameters
nn_dynamics(u, p, t) = ude_dynamics(u, p, t, p_)
# Define the problem
prob_nn = ODEProblem(nn_dynamics, X_n[:, 1], tspan, ComponentArray(p))

# Define a predictor
function predict(theta, X0 = X_n[:, 1], T = t)
    Array(solve(remake(prob_nn; u0 = X0, tspan = (T[1], T[end]), p = theta), Vern7();
        saveat = T, abstol = 1e-6, reltol = 1e-6))
end

# Simple L2 loss
loss(theta, _) = sum(abs2, X_n .- predict(theta))

# Container to track the losses
losses = Float64[]

callback = function (state, l)
    push!(losses, l)
    length(losses) % 50 == 0 && println("Current loss after ", length(losses), " iterations: ", losses[end])
    return false
end

## Usage of optimizers
We will use 2 optimizers in succession. Adam first moves the parameters into a favourable starting position, then BFGS converges quickly from there.

In [ ]:
optf = OptimizationFunction(loss, Optimization.AutoMooncake(; config = nothing))
optprob = OptimizationProblem(optf, ComponentArray(p))
res1 = solve(optprob, Adam(0.1); callback = callback, maxiters = 200)
println("Training loss after ", length(losses), " iterations: ", losses[end])
# Train with BFGS
optprob2 = OptimizationProblem(optf, res1.u)
res2 = solve(optprob2, Optim.BFGS(; initial_stepnorm = 0.01); callback = callback, maxiters = 1000)
println("Final training loss after ", length(losses), " iterations: ", losses[end])

# Rename the best candidate
p_trained = res2.u

# Plot the losses
pl_losses = plot(1:200, losses[1:200]; yaxis = :log10, xaxis = :log10, xlabel = "Iterations", ylabel = "Loss", label = "Adam", color = :blue)
plot!(201:length(losses), losses[201:end]; yaxis = :log10, xaxis = :log10, label = "BFGS", color = :red)

## Analysis of the trained network
Plot the data and the approximation.

In [ ]:
ts = first(solution.t):(mean(diff(solution.t)) / 2):last(solution.t)
X_hat = predict(p_trained, X_n[:, 1], ts)
# Trained on noisy data vs real solution
pl_trajectory = plot(ts, transpose(X_hat); xlabel = "t", ylabel = "x(t), y(t)", color = :red, label = ["UDE Approximation" nothing])
scatter!(solution.t, transpose(X_n); color = :black, label = ["Measurements" nothing])

The interactions the network learned, against the true ones, and the error.

In [ ]:
# Ideal unknown interactions of the predictor
Y_bar = [-p_[2] * (X_hat[1, :] .* X_hat[2, :])'; p_[3] * (X_hat[1, :] .* X_hat[2, :])']
# Neural network guess
Y_hat = first(U(X_hat, p_trained, st))

pl_reconstruction = plot(ts, transpose(Y_hat); xlabel = "t", ylabel = "U(x,y)", color = :red, label = ["UDE Approximation" nothing])
plot!(ts, transpose(Y_bar); color = :black, label = ["True Interaction" nothing])

# Plot the error
pl_reconstruction_error = plot(ts, norm.(eachcol(Y_bar - Y_hat)); yaxis = :log, xlabel = "t", ylabel = "L2-Error", label = nothing, color = :red)
pl_missing = plot(pl_reconstruction, pl_reconstruction_error; layout = (2, 1))
plot(pl_trajectory, pl_missing; size = (900, 450))

## Symbolic recovery of the missing terms
We will use the DataDrivenDiffEq.jl package to run symbolic recovery on the trained network. Through this, we will uncover the symbolic form which the neural network represents: sparse regression (SINDy) picks, among the polynomials in u1 and u2 up to degree 5, the few terms that reproduce the network's output.

In [ ]:
# Create a Basis
@variables u[1:2]
u = collect(u)
# Generate the basis functions: multivariate polynomials up to degree 5
basis = Basis(polynomial_basis(u, 5), u)

# Create the thresholds which should be used in the search process
lambda = exp10.(-3:0.1:0)
# Create an optimizer for the SINDy problem
opt = STLSQ(lambda)
# Create the problem for recovery: the network's output Y_hat at the states X_hat
nn_problem = DirectDataDrivenProblem(X_hat, Y_hat)
nn_res = solve(nn_problem, basis, opt; options = DataDrivenCommonOptions(; maxiters = 10_000))

# Show the results
recovered = get_basis(nn_res)
println(recovered)
println(get_parameter_map(recovered))
println("True: -", p_[2], " u1 u2 and ", p_[3], " u1 u2")

## The recovered model
The known linear terms with the recovered interactions, simulated from the start and compared with the truth.

In [ ]:
recovered_terms = (x, t) -> recovered(x, get_parameter_values(nn_res), t)
recovered_dynamics(x, p, t) = [p_[1] * x[1], -p_[4] * x[2]] .+ recovered_terms(x, t)
estimate = solve(ODEProblem(recovered_dynamics, u0, tspan), Tsit5(); saveat = solution.t)

plot(solution; color = :black, label = ["True" nothing])
plot!(estimate; color = :red, ls = :dash, label = ["Recovered" nothing])

## Discussion
We can see that the dynamics recovered through symbolic recovery match the original dynamics really well!